<a href="https://colab.research.google.com/github/NandanaRN/DeepLearning/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning Models and Applications

## Code structure for RNN
# Step 1: Install and Load the Dataset
Action: Install the Hugging Face datasets library and stream the data into memory.

Why: This pulls the dataset directly from the web without needing a manual download.
#Step 2: Clean and Normalize the Text
Action: Loop through the dataset to lowercase all strings and strip out special characters, punctuation, or numbers.

Why: This reduces vocabulary noise so your model doesn't treat terms like "World,", "world", and "world!" as three separate words.
#Step 3: Tokenize the Strings into Numbers
Action: Create a vocabulary lookup dictionary using a Tokenizer, assigning a unique integer ID to every unique word.

Why: Neural networks cannot interpret characters or letters; they only understand numbers and matrices.
#Step 4: Pad and Truncate Sequences
Action: Use pad_sequences to cap text lengths to a uniform size (e.g., exactly 50 words long). Shorter articles get trailing zeros; longer articles are trimmed down.

Why: Recurrent neural networks require the input tensor dimensions to be consistently shaped across the entire training batch.
#Step 5: Convert Labels to Categorical Vectors
Action: One-hot encode the target target variable labels. For instance, a class category label of 2 becomes [0, 0, 1, 0].

Why: This transforms the classification targets into a probability distribution format that directly matches what the model's terminal output layer will predict.
#Step 6: Define the RNN Model Architecture
Action: Stack three specific layers sequentially:
An Embedding Layer to translate integer IDs into dense, semantic vector spaces.
A SimpleRNN Layer to track sequential memory across the phrase.
A Dense Output Layer using a softmax activation to output classification scores.

Why: This establishes the pathway through which the text features are processed and compressed into predictions.
#Step 7: Compile and Train the Model
Action: Run model.compile() with an optimizer (like adam) and a classification loss function, then trigger model.fit() using your training data, defined epochs, and a validation split.

Why: This starts the active mathematical optimization loop where weights are adjusted iteratively based on prediction errors.
#Step 8: Read and Evaluate the Target Accuracy
Action: Monitor the logs generated at the end of each epoch or run a dedicated model.evaluate() step on your test subset.

Why: The console outputs individual metric readouts for accuracy (how well it fits training data) and val_accuracy (how well it performs on unseen validation data) so you can quantify the model's performance.

# STEP1

In [41]:
!pip install datasets

from datasets import load_dataset
import pandas as pd

#load the dataset (ag_news) from Hugging Face
raw_dataset=load_dataset("wangrongsheng/ag_news")

#convert the training split to a pandas dataframe for easy preprocessing
df=pd.DataFrame(raw_dataset["train"])
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


# Step 2: Clean and Normalize the Text

In [42]:
import re

def clean_text(text):
    text = text.lower()                      # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text)     # Remove punctuation, numbers, special characters
    return text

df["text"] = df["text"].apply(clean_text)

df.head()

,text,label
0,wall st bears claw back into the black reuters...,2
1,carlyle looks toward commercial aerospace reut...,2
2,oil and economy cloud stocks outlook reuters r...,2
3,iraq halts oil exports from main southern pipe...,2
4,oil prices soar to alltime record posing new m...,2


# Step 3: Tokenize the Text

In [43]:
from tensorflow.keras.preprocessing.text import Tokenizer

#imports tokenizer class
#this class converts raw text into numerical tokens
vocab_size = 10000

#top 10 000 most frequent words are kept .Here we set maximum size of the vocabulary.

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

#initializes the tokenizer.
#num_words restrict the dictionary size to the vocab_size.
#oov_token replaces any words not found in the top 10 000
#out of vocabulary is OOV


sequences = tokenizer.texts_to_sequences(df["text"])

#tranforms text string into a list of integers

# Step 4: Pad the Sequences

In [44]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
#import pad_sequence.
#purpose:it ensures all text sequnces in a dataset have identical lengths by adding zeroes or cutting off text
max_length = 50

#set the threshold variable to 50.(exactly 50 tokens long)

X = pad_sequences(sequences,
                  maxlen=max_length,
                  padding='post',
                  truncating='post')

print(X.shape)

#maxlen=max_length:output length to be 50.
#padding=post: add zeroes to the end of the sequence ,if it is shorter than 50.
#truncating=post:removes word from the end of the sequence if it is longer than 50.


(120000, 50)


# Step 5: Convert Labels

In [45]:
from tensorflow.keras.utils import to_categorical   #one hot encoding

y = to_categorical(df["label"])

print(y[:5])

[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


# Step 6: Split Data

In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,   #20% of testing and 80% for training
    random_state=42   #random generator
)



# Step 7: Build the RNN Model

In [47]:
from tensorflow.keras.models import Sequential
#linear stack of layer:sequential
#Think of Sequential as a way to build a neural network layer by layer, in a straight line, one after another.

from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

#Embedding: This part is like a magical translator.
#It takes each word from an article and turns it into a secret numerical code that the brain can understand.
#All similar words get similar codes.

#SimpleRNN: This is the 'memory' part of the brain.
#As it reads the secret codes of the words in a sentence, it tries to remember the whole idea of the sentence, not just individual words.
#It keeps a little note of what came before.

#Dense: This is the 'decision-maker' part.
#After the memory part has done its job, the decision-maker looks at all the remembered info and decides which of the four piles the article belongs in.

model = Sequential()  #empty sequential model

model.add(Embedding(input_dim=vocab_size,
                    output_dim=64,
                    input_length=max_length))

model.add(SimpleRNN(64))

#Dense(4, activation='softmax'): Predicts final multi-class classification probabilities.
#softmax: Normalizes outputs into a probability distribution.
#4: Classifies inputs into four distinct classes.

model.add(Dense(4, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Step 8: Compile the Model

In [48]:
model.compile(                            #initialize training
    optimizer='adam',                     #adjust the model automatically ,it speeds up learning.
    loss='categorical_crossentropy',      #used multiclass classification.
    metrics=['accuracy']                  #monitor accuracy
)

# Step 9: Train the Model

In [49]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,                #model goes through data 5 times.
    batch_size=32,           #model looks at 32 samples at once.
    validation_split=0.2     #holds back 20% of the data for testing.
)

Epoch 1/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 62s 25ms/step - accuracy: 0.7725 - loss: 0.6119 - val_accuracy: 0.8466 - val_loss: 0.4525
Epoch 2/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 52s 22ms/step - accuracy: 0.8838 - loss: 0.3631 - val_accuracy: 0.8680 - val_loss: 0.4383
Epoch 3/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 83s 22ms/step - accuracy: 0.9103 - loss: 0.2839 - val_accuracy: 0.8884 - val_loss: 0.3526
Epoch 4/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 81s 22ms/step - accuracy: 0.9230 - loss: 0.2420 - val_accuracy: 0.8954 - val_loss: 0.3180
Epoch 5/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 52s 22ms/step - accuracy: 0.9372 - loss: 0.1981 - val_accuracy: 0.8830 - val_loss: 0.3837


Step 10: Evaluate

In [50]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8803 - loss: 0.3931
Test Accuracy: 0.8802916407585144
